# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [22]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
#filter the table for product_name = 'pop'
pop_tart = cat[cat['product_name'].str.contains('pop', case=False)]
pop_tart

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
#list all SKUs
cat['sku']
#list corresponding SKU for product_name
cat['product_name']


,product_name
0,pop-tarts strawberry
1,Strawberry Pop-Tart
2,Pop-Tarts Strawberry


**Pop-Tart rows that are a different product:** All the items are strawberry pop-tarts.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [13]:
#build a dict from each SKU to map to PT-12
mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

#apply the mapping
cat['sku'] = cat['sku'].map(mapping).fillna(cat['sku'])
cat

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast
1,PT-12,strawberry pop-tart,Breakfast
2,PT-12,pop-tarts strawberry,Breakfast


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [14]:
#strip and title-case then show distinct values before and after counts
cat['category'] = cat['category'].str.strip().str.title()
cat['category'].value_counts()

#normalize category as breakfast
cat['category'] = cat['category'].replace('Breakfast Foods', 'Breakfast')
cat['category'].value_counts()

,count
category,
Breakfast,3


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [20]:
#lowercase and strip product name
cat['product_name'] = cat['product_name'].str.lower().str.strip()
cat['product_name']

#group by product name and show any group holding more than one SKU
grouped = cat.groupby('product_name').filter(lambda x: len(x) > 1)
grouped

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast
2,PT-12,pop-tarts strawberry,Breakfast


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [21]:
#assert that applying your mapping produced no nulls in the canonical column
assert cat['sku'].notnull().all()

#assert that applying your mapping did not change the number of rows
assert len(cat) == len(pop_tart)


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) The file did not run, so I was only able to see the pop-tart family based on the fallback data.

b) Consolidation becomes a business decision when it becomes unclear and not objective as to whether products belong in the same family. An example would be if there was a product that the analyst was unsure about belonging to the breakfast food group. They would need to consult the company or manufacturer of the product.